##  1D Convolution

In [2]:
%%writefile q9.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 60
#define LOW 10
#define HIGH 99
#define BLOCK_SIZE 8
#define MASK_WIDTH 5

// ── CUDA Kernel
__global__ void conv1DKernel(int *dN, int *dM, int *dP, int maskWidth, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        int dPvalue = 0;
        for (int j = 0; j < maskWidth; j++) {
            if ((i + j) < n)
                dPvalue += dN[i + j] * dM[maskWidth - 1 - j]; // flipped mask
        }
        dP[i] = dPvalue;
    }
}

// ── Host Function
__host__ void conv1D(int *hN, int *hM, int *hP, int maskWidth, int n)
{
    int *dN, *dM, *dP;
    int sizeN = n * sizeof(int);
    int sizeM = maskWidth * sizeof(int);

    cudaMalloc((void**)&dN, sizeN);
    cudaMalloc((void**)&dM, sizeM);
    cudaMalloc((void**)&dP, sizeN);

    cudaMemcpy(dN, hN, sizeN, cudaMemcpyHostToDevice);
    cudaMemcpy(dM, hM, sizeM, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    conv1DKernel<<<DimGrid, DimBlock>>>(dN, dM, dP, maskWidth, n);

    cudaMemcpy(hP, dP, sizeN, cudaMemcpyDeviceToHost);

    cudaFree(dN); cudaFree(dM); cudaFree(dP);
}

// ── Main
int main()
{
    int hN[N], hM[MASK_WIDTH], hP[N];

    srand(time(NULL));
    for (int i = 0; i < N; i++)
        hN[i] = rand() % (HIGH - LOW + 1) + LOW;

    // Fixed mask
    int mask[MASK_WIDTH] = {3, 4, 5, 4, 3};
    for (int i = 0; i < MASK_WIDTH; i++)
        hM[i] = mask[i];

    printf("hN:\n");
    for (int i = 0; i < N; i++) printf("%4d", hN[i]);
    printf("\n");

    printf("\nhM (Mask):\n");
    for (int i = 0; i < MASK_WIDTH; i++) printf("%4d", hM[i]);
    printf("\n");

    conv1D(hN, hM, hP, MASK_WIDTH, N);

    printf("\nhP (1D Convolution Result):\n");
    for (int i = 0; i < N; i++) printf("%6d", hP[i]);
    printf("\n");

    return 0;
}

Writing q9.cu


In [3]:
!nvcc -arch=sm_75 q9.cu -o q9

!./q9

hN:
  75  50  71  26  25  55  11  16  98  68  44  65  28  13  30  97  10  59  22  50  63  27  43  16  56  98  92  62  44  19  35  71  21  58  87  37  65  88  95  64  19  40  81  37  95  63  35  95  74  99  45  90  79  40  96  35  38  50  88  34

hM (Mask):
   3   4   5   4   3

hP (1D Convolution Result):
   959   829   695   545   708   885   991  1151  1130   856   674   783   704   861   830   864   765   848   818   757   744   851  1141  1316  1400  1205   925   810   730   810   987  1046  1073  1231  1371  1383  1335  1160  1039   903  1055  1198  1223  1231  1314  1397  1386  1526  1440  1363  1338  1275  1131   981  1082   961   890   672   400   102
